# EDS Colab Processing Server
## Emotion Data Studio — GPU Processing on Google Colab

**Chay tren Colab Pro (GPU T4/V100/A100)**: Pipeline AI toc do cao, truy cap dashboard tu trinh duyet bat ky dau qua ngrok.

**Sau khi chay xong**: Dashboard se co URL ngrok o cell cuoi cung.

In [ ]:
# ============================================================
# 0. CAI DAT MOI TRUONG
# ============================================================

# Upgrade pip
!pip install -q --upgrade pip

# Install core dependencies
!pip install -q \
  fastapi uvicorn \
  sqlalchemy pydantic pydantic-settings \
  transformers datasets accelerate \
  torch torchaudio \
  librosa opencv-python \
  yt-dlp gdown \
  google-cloud-storage \
  psycopg2-binary pg8000 \
  pyngrok httpx \
  google-cloud-aiplatform>=2.0.0

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU — processing will be slow. Use a GPU runtime.")


In [ ]:
# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
EDS_ROOT = '/content/drive/MyDrive/EDS'
os.makedirs(EDS_ROOT, exist_ok=True)
os.makedirs(f'{EDS_ROOT}/videos', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/clips', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/audio', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/features', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/exports', exist_ok=True)
os.makedirs(f'{EDS_ROOT}/data', exist_ok=True)

print(f"Drive mounted: {EDS_ROOT}")

In [ ]:
# ============================================================
# 2. CLONE / PULL EDS REPO
# ============================================================

REPO_URL = "https://github.com/YOUR_USERNAME/BCDA.git"  # <-- THAY DOI
REPO_DIR = '/content/BCDA'
EDS_DIR = f'{REPO_DIR}/tools/emotion-data-studio'

import subprocess, os

if os.path.exists(REPO_DIR):
    print("Repo da ton tai — pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=False)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

import sys
# backend/ is inside tools/emotion-data-studio/, so add EDS_DIR first
sys.path.insert(0, EDS_DIR)
sys.path.insert(0, REPO_DIR)

print(f"Repo ready: {REPO_DIR}")

In [ ]:
# ============================================================
# 3. CAU HINH GOOGLE CLOUD & GEMINI AUTO-LABELER
# ============================================================
#
# 1. Tao service account tai: https://console.cloud.google.com/iam-admin/serviceaccounts
#    Roles: Storage Admin + Vertex AI User
# 2. Tai JSON key -> upload len Drive: /content/drive/MyDrive/EDS/credentials/
# 3. Tao GCS bucket: gs://your-bucket-name
# 4. Dien GCP_PROJECT_ID, GCS_BUCKET_NAME ben duoi

import os

SERVICE_ACCOUNT_KEY = "/content/drive/MyDrive/EDS/credentials/service-account.json"
if os.path.exists(SERVICE_ACCOUNT_KEY):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SERVICE_ACCOUNT_KEY
    print(f"Service account: OK")
else:
    print("WARNING: Service account key chua co!")

os.environ["GCP_PROJECT_ID"] = "your-gcp-project-id"
os.environ["GCP_LOCATION"] = "us-central1"
os.environ["GCS_BUCKET_NAME"] = "your-bucket-name-emotion-data"

os.environ["GEMINI_MODEL"] = "gemini-2.5-flash"
os.environ["GEMINI_TEMPERATURE"] = "0.2"
os.environ["GEMINI_MAX_TOKENS"] = "8192"
os.environ["GEMINI_INTENSITY_THRESHOLD"] = "0.6"

os.environ["EDS_DATA_DIR"] = f"{EDS_ROOT}/data"
os.environ["EDS_DOWNLOAD_MODE"] = "balanced"
os.environ["EDS_DOWNLOAD_MAX_HEIGHT"] = "720"

!pip install -q google-cloud-aiplatform>=2.0.0

try:
    from backend.services.gemini_auto_labeler import GeminiAutoLabeler
    labeler = GeminiAutoLabeler()
    print(f"Gemini: {labeler.status()}")
except Exception as e:
    print(f"WARNING: Gemini config error: {e}")

print("Google Cloud & Gemini configured")

In [ ]:
# ============================================================
# 4. PREWARM MODELS (GPU)
# ============================================================

import sys, logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('EDS')

# Init database
from backend.database.local_db import init_database, get_session
init_database()
print("Database initialized")

# Prewarm core models (skip text/audio emotion for now — loaded lazily)
from backend.ai_models.model_manager import model_manager

logger.info("Prewarming Whisper (medium)...")
try:
    model_manager.prewarm_models(['whisper', 'deepface', 'mtcnn'])
    logger.info("Core models loaded")
except Exception as e:
    logger.warning(f"Some models failed: {e}")

# Load text/audio emotion models (optional)
logger.info("Loading text/audio emotion models...")
try:
    model_manager.prewarm_models(['text_emotion', 'audio_emotion'])
    logger.info("Emotion models loaded")
except Exception as e:
    logger.warning(f"Emotion models failed: {e}")

print(model_manager.status())

In [ ]:
# ============================================================
# 5. KHOI DONG WEB DASHBOARD
# ============================================================

import subprocess, threading, time

# Start ngrok in background
# 1. Get ngrok auth token: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "YOUR_NGROK_TOKEN"  # <-- THAY DOI voi token that cua ban

if NGROK_TOKEN == "YOUR_NGROK_TOKEN":
    print("WARNING: Chua dat NGROK_TOKEN! Dashboard se chay o port 8765 (khong public).")
    print("   Lay token tai: https://dashboard.ngrok.com/get-started/your-authtoken")
    public_url = None
else:
    get_ipython().system('pip install -q pyngrok')
    from pyngrok import ngrok
    
    # Kill any existing tunnels
    ngrok.kill()
    
    # Set token
    ngrok.set_auth_token(NGROK_TOKEN)
    
    # Create tunnel to port 8765
    tunnel = ngrok.connect(addr="8765", proto="http", bind_tls=True)
    public_url = tunnel.public_url
    print(f"Dashboard: {public_url}")

# Start FastAPI server
import os
os.chdir(f'{REPO_DIR}/tools/emotion-data-studio')

def run_server():
    import uvicorn
    uvicorn.run(
        "web.main:app",
        host="0.0.0.0",
        port=8765,
        log_level="info",
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(3)
print("Web server started on port 8765")

In [ ]:
# ============================================================
# 6. START PIPELINE WORKER (BACKGROUND)
# ============================================================

import threading, time

def run_pipeline():
    import os
    os.environ.setdefault('EDS_DATA_DIR', f'{EDS_ROOT}/data')
    os.environ.setdefault('EDS_VIDEO_DIR', f'{EDS_ROOT}/videos')
    os.environ.setdefault('EDS_CLIP_DIR', f'{EDS_ROOT}/clips')
    os.environ.setdefault('EDS_AUDIO_DIR', f'{EDS_ROOT}/audio')
    os.environ.setdefault('EDS_FEATURE_DIR', f'{EDS_ROOT}/features')

    from backend.services.pipeline_orchestrator import pipeline_orchestrator
    pipeline_orchestrator.run_until_complete()

worker = threading.Thread(target=run_pipeline, daemon=True)
worker.start()

print("Pipeline worker started")

In [ ]:
# ============================================================
# 7. HUONG DAN SU DUNG
# ============================================================

print("=" * 60)
print("EMOTION DATA STUDIO — COLAB READY")
print("=" * 60)
print()
if 'public_url' in dir() and public_url:
    print(f"DASHBOARD: {public_url}")
    print()
print("Cac buoc tiep theo:")
print("1. Mo dashboard tren trinh duyet")
print("2. Nhap video URL de thu hoach")
print("3. Theo doi tien trinh xu ly")
print("4. Review va xuat dataset")
print()
print(f"Data dir: {EDS_ROOT}")
print(f"Videos:   {EDS_ROOT}/videos")
print(f"Clips:    {EDS_ROOT}/clips")
print(f"Exports:  {EDS_ROOT}/exports")